In [1]:
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"[INFO] Using {device} device")

[INFO] Using cuda device


In [3]:
dataset = load_dataset("argilla/news-summary")

dataset = dataset.remove_columns(
    ['prediction', 'prediction_agent', 'annotation', 'annotation_agent', 'metadata', 'status', 'event_timestamp', 'metrics', 'id']
)

In [4]:
dataset_test = dataset["test"]

In [5]:
dataset_test[0]['text']

'WASHINGTON (Reuters) - President Donald Trump on Tuesday scrapped an Obama-era program that protects from deportation immigrants brought illegally into the United States as children, delaying implementation until March and giving a gridlocked Congress six months to decide the fate of almost 800,000 young people. As the so-called Dreamers who have benefited from the five-year-old program were plunged into uncertainty, business and religious leaders, mayors, governors, Democratic lawmakers, unions, civil liberties advocates and former Democratic President Barack Obama all condemned Trump’s move. The action was announced not by Trump but by Jeff Sessions, his attorney general, who called the Deferred Action for Childhood Arrivals (DACA) program an unconstitutional overreach by Obama. There will be an “orderly, lawful wind-down,” Sessions said. Trump later issued a written statement saying that “I do not favor punishing children, most of whom are now adults, for the actions of their paren

In [6]:
sentiment_analysis = pipeline('text-classification', 'cardiffnlp/twitter-roberta-base-sentiment-latest')

cola_tokenizer = AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
cola_model = AutoModelForSequenceClassification.from_pretrained("textattack/roberta-base-CoLA")
linguistic_acceptability = pipeline('text-classification', model=cola_model, tokenizer=cola_tokenizer)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0
Some weights of the model checkpoint at textattack/roberta-base-CoLA were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceCla

In [7]:
def neutral_scores(texts):
    scores = []
    results = sentiment_analysis(texts, function_to_apply='none', top_k=None)
    for result in results:
        for label in result:
            if label['label'] == 'neutral':
                scores.append(label['score'])
    return scores

def linguistic_acceptable_scores(texts):
    scores = []
    results = linguistic_acceptability(texts, function_to_apply='none', top_k=None)
    for result in results:
        for label in result:
            if label['label'] == 'LABEL_1':
                scores.append(label['score'])
    return scores

In [8]:
flan_t5_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
origin_summarizer = pipeline("summarization", "google/flan-t5-small", tokenizer=flan_t5_tokenizer)

Device set to use cuda:0


In [9]:
rl_tokenizer = AutoTokenizer.from_pretrained("flan-t5-rl")
rl_summarizer = pipeline("summarization", 'flan-t5-rl', tokenizer=rl_tokenizer)

Some weights of the model checkpoint at flan-t5-rl were not used when initializing T5ForConditionalGeneration: ['v_head.summary.bias', 'v_head.summary.weight']
- This IS expected if you are initializing T5ForConditionalGeneration from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing T5ForConditionalGeneration from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


In [10]:
generation_kwargs = {
    #"min_length": -1, # don't ignore the EOS token
    "min_length": 3,
    "top_k": 0.0, # no top-k sampling
    "top_p": 1.0, # no nucleus sampling
    "do_sample": True,
    "pad_token_id": flan_t5_tokenizer.eos_token_id,
    #"max_length": 70,
    "max_new_tokens": 70,
}

In [11]:
# input_data = dataset_test[0]['text']
orig_neutral_score_list, orig_cola_score_list, rl_neutral_score_list, rl_cola_score_list = [], [], [], []
test_range = 100

for idx in range(test_range):
    input_data = dataset_test[idx]['text']
    print(f'@@@ Summarize {idx} @@@')

    for s in range(2):
        if s == 0:
            print("========== Using original summarizer ==========")
        else:
            print("========== Using reinforced summarizer ==========")
    
        neutral_score_list, cola_score_list = [], []
    
        for i in range(10):
            results = origin_summarizer(input_data, **generation_kwargs) if s == 0 else rl_summarizer(input_data, **generation_kwargs)
            summary_text = results[0]['summary_text']
        
            if summary_text.endswith('.') and '...' not in summary_text:
                #print(f"Summary {i+1}: {summary_text}")
        
                neutral_score = neutral_scores([summary_text])
                neutral_score_list.extend(neutral_score)
                #print(f"Summary {i+1}: Neutral score : {neutral_score}")
        
                cola_score = linguistic_acceptable_scores([summary_text])
                cola_score_list.extend(cola_score)
                #print(f"Summary {i+1}: Cola score: {cola_score}")
        
                #print('############')

        if len(neutral_score_list) == 0 | len(cola_score_list) == 0:
            continue

        neutral_score_avg = sum(neutral_score_list) / len(neutral_score_list)
        cola_score_avg = sum(cola_score_list) / len(cola_score_list)
        
        print(f'=> Average Neutral Score: {neutral_score_avg}')
        print(f'=> Average Cola Score: {cola_score_avg}')
        print('=================================================')

        if s == 0:
            orig_neutral_score_list.append(neutral_score_avg)
            orig_cola_score_list.append(cola_score_avg)
        else:
            rl_neutral_score_list.append(neutral_score_avg)
            rl_cola_score_list.append(cola_score_avg)

orig_neutral_score_avg = sum(orig_neutral_score_list) / len(orig_neutral_score_list)
orig_cola_score_avg = sum(orig_cola_score_list) / len(orig_cola_score_list)

rl_neutral_score_avg = sum(rl_neutral_score_list) / len(rl_neutral_score_list)
rl_cola_score_avg = sum(rl_cola_score_list) / len(rl_cola_score_list)

print(f'==> Origin Modle Average Neutral Score: {orig_neutral_score_avg}; Average Cola Score: {orig_cola_score_avg}')
print(f'==> RL Model Average Neutral Score: {rl_neutral_score_avg}; Average Cola Score: {rl_cola_score_avg}')

Token indices sequence length is longer than the specified maximum sequence length for this model (1500 > 512). Running this sequence through the model will result in indexing errors


@@@ Summarize 0 @@@
========== Using original summarizer ==========


Token indices sequence length is longer than the specified maximum sequence length for this model (1500 > 512). Running this sequence through the model will result in indexing errors


=> Average Neutral Score: 0.3726331055164337
=> Average Cola Score: 1.4025773286819458
========== Using reinforced summarizer ==========


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


...

=> Average Neutral Score: 0.13305236399173737
=> Average Cola Score: 0.9147466818491617
@@@ Summarize 90 @@@
========== Using original summarizer ==========


Your max_length is set to 200, but your input_length is only 159. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=79)
Your max_length is set to 200, but your input_length is only 159. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=79)
Your max_length is set to 200, but your input_length is only 159. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=79)
Your max_length is set to 200, but your input_length is only 159. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=79)


=> Average Neutral Score: 0.9159414768218994
=> Average Cola Score: 2.006007671356201
========== Using reinforced summarizer ==========


Your max_length is set to 200, but your input_length is only 159. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=79)
Your max_length is set to 200, but your input_length is only 159. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=79)
Your max_length is set to 200, but your input_length is only 159. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=79)
Your max_length is set to 200, but your input_length is only 159. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=79)


=> Average Neutral Score: 0.8332895159721374
=> Average Cola Score: 2.0036343336105347
@@@ Summarize 91 @@@
========== Using original summarizer ==========
=> Average Neutral Score: 1.7144247889518738
=> Average Cola Score: 1.7671012083689372
========== Using reinforced summarizer ==========
=> Average Neutral Score: 1.8649444580078125
=> Average Cola Score: 1.812699794769287
@@@ Summarize 92 @@@
========== Using original summarizer ==========
=> Average Neutral Score: 1.0561221341292064
=> Average Cola Score: 1.8883335987726848
========== Using reinforced summarizer ==========
=> Average Neutral Score: 0.9470282878194537
=> Average Cola Score: 1.9399346794400896
@@@ Summarize 93 @@@
========== Using original summarizer ==========
=> Average Neutral Score: 1.4998119831085206
=> Average Cola Score: 0.975537559390068
========== Using reinforced summarizer ==========
=> Average Neutral Score: 1.7166159629821778
=> Average Cola Score: 0.8682888170704246
@@@ Summarize 94 @@@
========== Usin